# MCP Servers in Agentic AI: Complete Hands-On Learning Path

## Overview
This notebook teaches you everything about **Model Context Protocol (MCP) Servers** - from basics to deploying production agents.

### What You'll Learn:
1. 🏗️ Build custom MCP servers
2. 🔌 Integrate pre-built servers (Azure DevOps, SonarQube, Microsoft Azure)
3. 🤖 Create agentic AI workflows
4. 🚀 Deploy and test MCP solutions
5. 🛡️ Add security and authentication

**Target Audience:** Beginners with basic Python knowledge

**Time Estimate:** 2-3 hours for full completion

---

## Prerequisites
- Python 3.8+
- Basic understanding of APIs and async programming
- Familiarity with JSON and REST concepts


---

# Section 1: Dev Environment Setup for MCP Development
Setup Python dependencies and create a project structure for MCP development.

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    "pydantic>=2.0",
    "json-rpc>=1.14.1",
    "httpx>=0.24.0",
    "aiofiles>=23.0.0",
    "python-dotenv>=1.0.0",
    "openai>=1.0.0",  # For agent examples
    "pymongo>=4.5.0",  # For database examples (optional)
    "sqlalchemy>=2.0.0",  # For SQL examples
    "pandas>=2.0.0",
    "requests>=2.31.0"
]

print("Installing MCP development dependencies...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package}")
    except Exception as e:
        print(f"⚠ {package}: {str(e)[:50]}")

print("\n✓ Environment setup complete!")

In [ ]:
# Import libraries and setup
import json
import asyncio
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✓ All imports successful!")
print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {os.getcwd()}")

---

# Section 2: Understanding MCP Architecture

## What is MCP?

**MCP (Model Context Protocol)** is a standardized interface for connecting AI models to data, tools, and services.

### Key Components:

1. **MCP Client** - Your AI application that uses MCP servers
2. **MCP Server** - Exposes tools, resources, and prompts
3. **Tool** - An executable function that performs an action
4. **Resource** - Data (files, documents, database records)
5. **Prompt** - Pre-defined instruction templates

### Request/Response Pattern:

```
Client → [JSON-RPC Request] → Server
         [tool name + arguments]
         
Server → [JSON-RPC Response] → Client
         [result + data]
```

Let's visualize this:


In [ ]:
# Visualize MCP Architecture
import json

# Example MCP JSON-RPC request
example_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "search_documents",
        "arguments": {
            "query": "deployment strategies",
            "max_results": 5
        }
    }
}

# Example MCP JSON-RPC response
example_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "content": [
            {
                "type": "text",
                "text": "Found 3 documents matching 'deployment strategies'"
            }
        ]
    }
}

print("📤 EXAMPLE MCP REQUEST:")
print(json.dumps(example_request, indent=2))
print("\n" + "="*60 + "\n")
print("📥 EXAMPLE MCP RESPONSE:")
print(json.dumps(example_response, indent=2))

# Data classes for MCP structures
@dataclass
class MCPTool:
    name: str
    description: str
    input_schema: Dict[str, Any]
    
@dataclass
class MCPResource:
    uri: str
    name: str
    mime_type: str
    
@dataclass
class MCPResponse:
    content: List[Dict[str, Any]]
    error: Optional[str] = None

print("\n" + "="*60)
print("✓ MCP data structures defined")


---

# Section 3: Build Your First Minimal MCP Server

Let's create a simple MCP server with:
- ✅ One **Tool** (function)
- ✅ One **Resource** (data)
- ✅ JSON-RPC communication pattern

This is the foundation for all MCP servers!


In [ ]:
class MinimalMCPServer:
    """
    A minimal MCP Server implementation with:
    - One Tool: calculator
    - One Resource: server_info
    """
    
    def __init__(self, name: str = "minimal-server"):
        self.name = name
        self.tools = {
            "add": {
                "description": "Add two numbers",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "number"},
                        "b": {"type": "number"}
                    },
                    "required": ["a", "b"]
                }
            },
            "multiply": {
                "description": "Multiply two numbers",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "number"},
                        "b": {"type": "number"}
                    },
                    "required": ["a", "b"]
                }
            }
        }
        
        self.resources = {
            "server://info": {
                "name": "Server Information",
                "mime_type": "application/json",
                "data": {
                    "server_name": self.name,
                    "version": "1.0.0",
                    "tools_available": list(self.tools.keys()),
                    "created_at": datetime.now().isoformat()
                }
            }
        }
    
    def list_tools(self):
        """Return all available tools"""
        return {
            "tools": [
                {"name": name, **details}
                for name, details in self.tools.items()
            ]
        }
    
    def list_resources(self):
        """Return all available resources"""
        return {
            "resources": [
                {"uri": uri, "name": data["name"], "mimeType": data["mime_type"]}
                for uri, data in self.resources.items()
            ]
        }
    
    def read_resource(self, uri: str):
        """Read a resource by URI"""
        if uri in self.resources:
            return {"content": self.resources[uri]["data"]}
        return {"error": f"Resource not found: {uri}"}
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Execute a tool with given arguments"""
        if tool_name == "add":
            result = arguments["a"] + arguments["b"]
            return {"result": result, "operation": "add"}
        elif tool_name == "multiply":
            result = arguments["a"] * arguments["b"]
            return {"result": result, "operation": "multiply"}
        else:
            return {"error": f"Tool not found: {tool_name}"}
    
    def handle_request(self, request: Dict[str, Any]):
        """Handle a JSON-RPC request"""
        method = request.get("method")
        params = request.get("params", {})
        request_id = request.get("id")
        
        result = None
        
        if method == "tools/list":
            result = self.list_tools()
        elif method == "resources/list":
            result = self.list_resources()
        elif method == "resources/read":
            result = self.read_resource(params.get("uri"))
        elif method == "tools/call":
            result = self.call_tool(params.get("name"), params.get("arguments", {}))
        else:
            result = {"error": f"Unknown method: {method}"}
        
        return {
            "jsonrpc": "2.0",
            "id": request_id,
            "result": result
        }

# Create and test the server
server = MinimalMCPServer("my-first-server")

print("✓ Minimal MCP Server created!")
print(f"Server name: {server.name}")


In [ ]:
# Test the minimal MCP server
print("\n" + "="*60)
print("TEST 1: List available tools")
print("="*60)
response = server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list"
})
print(json.dumps(response, indent=2))

print("\n" + "="*60)
print("TEST 2: List available resources")
print("="*60)
response = server.handle_request({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "resources/list"
})
print(json.dumps(response, indent=2))

print("\n" + "="*60)
print("TEST 3: Call tool (add)")
print("="*60)
response = server.handle_request({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "add",
        "arguments": {"a": 10, "b": 5}
    }
})
print(json.dumps(response, indent=2))

print("\n" + "="*60)
print("TEST 4: Read resource")
print("="*60)
response = server.handle_request({
    "jsonrpc": "2.0",
    "id": 4,
    "method": "resources/read",
    "params": {"uri": "server://info"}
})
print(json.dumps(response, indent=2))


---

# Section 4: MCP Architecture in Agentic AI Workflows

An **Agent** is an AI system that:
1. **Perceives** environment (reads from MCP resources)
2. **Reasons** about the data (using LLM)
3. **Acts** on the world (calls MCP tools)
4. **Observes** results (receives feedback)

### Agent Loop Flow:

```
┌─────────────────────────────────────────┐
│  1. Agent perceives state via MCP       │
│     (reads resources, context)          │
└────────────┬────────────────────────────┘
             │
┌────────────▼────────────────────────────┐
│  2. LLM reasons about situation         │
│     (decides which tool to use)         │
└────────────┬────────────────────────────┘
             │
┌────────────▼────────────────────────────┐
│  3. Agent calls MCP tool                │
│     (executes action)                   │
└────────────┬────────────────────────────┘
             │
┌────────────▼────────────────────────────┐
│  4. MCP returns result                  │
│     (feedback to agent)                 │
└────────────┬────────────────────────────┘
             │
             └──→ Loop until goal achieved
```

Let's model this:


In [ ]:
# Model Agentic AI with MCP Architecture
class SimpleAgent:
    """
    A simple agent that:
    1. Reads context from MCP servers
    2. Makes decisions with LLM simulation
    3. Calls MCP tools
    4. Observes and updates knowledge
    """
    
    def __init__(self, name: str, mcp_servers: List[MinimalMCPServer] = None):
        self.name = name
        self.mcp_servers = mcp_servers or []
        self.memory = {
            "interactions": [],
            "current_goal": None,
            "observations": []
        }
    
    def add_mcp_server(self, server: MinimalMCPServer):
        """Register an MCP server"""
        self.mcp_servers.append(server)
    
    def perceive(self):
        """STEP 1: Agent perceives environment via MCP"""
        perception = {
            "timestamp": datetime.now().isoformat(),
            "servers": len(self.mcp_servers),
            "available_tools": [],
            "resources": []
        }
        
        # Discover tools and resources from all MCP servers
        for server in self.mcp_servers:
            tools_response = server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "tools/list"
            })
            perception["available_tools"].extend(
                [t["name"] for t in tools_response["result"]["tools"]]
            )
            
            resources_response = server.handle_request({
                "jsonrpc": "2.0",
                "id": 2,
                "method": "resources/list"
            })
            perception["resources"].extend(
                [r["uri"] for r in resources_response["result"]["resources"]]
            )
        
        return perception
    
    def reason(self, perception: Dict, goal: str) -> str:
        """STEP 2: Agent reasons about what to do (simulate LLM)"""
        # In real systems, this would call Claude/GPT-4
        # For demo, we'll implement simple heuristics
        
        if "add" in goal.lower():
            return "add"
        elif "multiply" in goal.lower():
            return "multiply"
        else:
            return "unknown"
    
    def act(self, tool_name: str, arguments: Dict) -> Dict:
        """STEP 3: Agent calls MCP tool"""
        for server in self.mcp_servers:
            response = server.handle_request({
                "jsonrpc": "2.0",
                "id": 3,
                "method": "tools/call",
                "params": {
                    "name": tool_name,
                    "arguments": arguments
                }
            })
            if "error" not in response.get("result", {}):
                return response["result"]
        
        return {"error": f"Tool not found: {tool_name}"}
    
    def observe(self, result: Dict):
        """STEP 4: Agent observes result and updates memory"""
        self.memory["observations"].append({
            "timestamp": datetime.now().isoformat(),
            "result": result
        })
    
    def run_goal(self, goal: str, data: Dict):
        """Run the full agent loop"""
        print(f"\n🤖 Agent: {self.name}")
        print(f"📍 Goal: {goal}")
        print("-" * 60)
        
        # Step 1: Perceive
        print("1️⃣  PERCEIVE: Agent reads context from MCP servers...")
        perception = self.perceive()
        print(f"   Found {len(perception['available_tools'])} tools: {perception['available_tools']}")
        
        # Step 2: Reason
        print("2️⃣  REASON: Agent decides which tool to use...")
        tool_choice = self.reason(perception, goal)
        print(f"   Chosen tool: {tool_choice}")
        
        # Step 3: Act
        print("3️⃣  ACT: Agent calls MCP tool...")
        result = self.act(tool_choice, data)
        print(f"   Result: {result}")
        
        # Step 4: Observe
        print("4️⃣  OBSERVE: Agent updates memory...")
        self.observe(result)
        print("   ✓ Memory updated")
        
        return result

# Create an agent and test it
agent = SimpleAgent("TaskBot")
agent.add_mcp_server(server)

# Test agent with a goal
result = agent.run_goal("add two numbers", {"a": 7, "b": 3})
print("\n✓ Agent completed task!")


---

# Section 5: Build MCP Tool for Document Q&A (RAG)

**RAG (Retrieval-Augmented Generation)** combines:
1. **Retrieval**: Find relevant documents
2. **Augmentation**: Add documents to LLM context
3. **Generation**: LLM generates answer with context

This MCP tool enables agents to query documents intelligently.


In [ ]:
# Simple RAG (Document Q&A) MCP Server
class DocumentRAGMCPServer(MinimalMCPServer):
    """
    MCP Server for Retrieval-Augmented Generation
    - Stores documents
    - Provides search capability
    - Returns relevant context for queries
    """
    
    def __init__(self):
        super().__init__("document-rag-server")
        
        # Sample documents (in real system, from database)
        self.documents = [
            {
                "id": "doc1",
                "title": "Azure DevOps CI/CD Basics",
                "content": """
                Azure DevOps provides integrated development tools for planning, developing, 
                deploying, and maintaining software. Key features include:
                - Pipelines for CI/CD automation
                - Repos for version control
                - Boards for project management
                - Test Plans for quality assurance
                """,
                "category": "azure"
            },
            {
                "id": "doc2",
                "title": "GitHub Actions Workflows",
                "content": """
                GitHub Actions automate software workflows in your repository with CI/CD.
                Workflows are defined in YAML files:
                - Triggers: push, pull_request, schedule
                - Jobs: run tests, build, deploy
                - Actions: reusable units of work
                """,
                "category": "github"
            },
            {
                "id": "doc3",
                "title": "SonarQube Code Quality",
                "content": """
                SonarQube analyzes code to detect bugs, security issues, and technical debt.
                Key metrics:
                - Code coverage: percentage of code tested
                - Bugs: defects found in code
                - Vulnerabilities: security issues
                - Code smells: maintainability problems
                """,
                "category": "sonarqube"
            }
        ]
        
        # Update tools and resources
        self.tools["search_documents"] = {
            "description": "Search documents by keyword or category",
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "category": {"type": "string", "default": "all"}
                },
                "required": ["query"]
            }
        }
        
        self.resources["documents://all"] = {
            "name": "All Documents",
            "mime_type": "application/json",
            "data": {"total": len(self.documents), "docs": self.documents}
        }
    
    def search_documents(self, query: str, category: str = "all"):
        """Simple keyword search (in real system, use embeddings/semantic search)"""
        query_lower = query.lower()
        results = []
        
        for doc in self.documents:
            if category != "all" and doc["category"] != category:
                continue
            
            # Check if query matches title or content
            if query_lower in doc["title"].lower() or query_lower in doc["content"].lower():
                results.append({
                    "id": doc["id"],
                    "title": doc["title"],
                    "score": 0.95,  # In real system, similarity score
                    "excerpt": doc["content"][:200] + "..."
                })
        
        return {
            "query": query,
            "category": category,
            "total_results": len(results),
            "results": results
        }
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Override to add search_documents tool"""
        if tool_name == "search_documents":
            return self.search_documents(
                arguments.get("query"),
                arguments.get("category", "all")
            )
        else:
            return super().call_tool(tool_name, arguments)

# Create RAG server
rag_server = DocumentRAGMCPServer()

print("✓ RAG MCP Server created!")
print(f"Documents available: {len(rag_server.documents)}")

# Test search
print("\n" + "="*60)
print("TEST: Search documents for 'CI/CD'")
print("="*60)
response = rag_server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "search_documents",
        "arguments": {"query": "CI/CD"}
    }
})
print(json.dumps(response["result"], indent=2))


---

# Section 6: Build MCP Tool for Database Access (SQL with Guardrails)

Safely expose database queries through MCP with:
- ✅ Read-only constraints
- ✅ Parameterized queries (prevent SQL injection)
- ✅ Result limits (prevent data exfiltration)
- ✅ Audit logging


In [ ]:
# Database MCP Server with Guardrails
class DatabaseMCPServer(MinimalMCPServer):
    """
    MCP Server for safe database access with:
    - Read-only enforcement
    - Parameterized queries
    - Row limits
    - Audit logging
    """
    
    def __init__(self):
        super().__init__("database-server")
        
        # Simulated database tables
        self.tables = {
            "employees": [
                {"id": 1, "name": "Alice", "department": "Engineering", "salary": 120000},
                {"id": 2, "name": "Bob", "department": "Sales", "salary": 80000},
                {"id": 3, "name": "Charlie", "department": "Engineering", "salary": 130000},
            ],
            "projects": [
                {"id": 101, "name": "Project Alpha", "status": "active", "team_size": 5},
                {"id": 102, "name": "Project Beta", "status": "planning", "team_size": 3},
            ]
        }
        
        # Audit log
        self.audit_log = []
        
        # Update tools
        self.tools["query_table"] = {
            "description": "Query a table with optional filters (READ-ONLY)",
            "input_schema": {
                "type": "object",
                "properties": {
                    "table": {"type": "string", "enum": ["employees", "projects"]},
                    "where_column": {"type": "string"},
                    "where_value": {"type": "string"},
                    "limit": {"type": "integer", "default": 10}
                },
                "required": ["table"]
            }
        }
        
        self.tools["get_table_schema"] = {
            "description": "Get schema/columns of a table",
            "input_schema": {
                "type": "object",
                "properties": {
                    "table": {"type": "string", "enum": ["employees", "projects"]}
                },
                "required": ["table"]
            }
        }
    
    def query_table(self, table: str, where_column: str = None, where_value: str = None, limit: int = 10):
        """Safe parameterized query"""
        # Enforce limit
        limit = min(limit, 50)  # Max 50 rows
        
        if table not in self.tables:
            self._log_query(table, where_column, where_value, "FAILED", "Table not found")
            return {"error": f"Table not found: {table}"}
        
        data = self.tables[table]
        
        # Apply filter if provided
        if where_column and where_value:
            filtered = [row for row in data if str(row.get(where_column)) == where_value]
            data = filtered
        
        # Apply limit
        data = data[:limit]
        
        # Log query
        self._log_query(table, where_column, where_value, "SUCCESS", f"Returned {len(data)} rows")
        
        return {
            "table": table,
            "rows": len(data),
            "data": data,
            "limit_applied": limit
        }
    
    def get_table_schema(self, table: str):
        """Get columns and types"""
        if table not in self.tables:
            return {"error": f"Table not found: {table}"}
        
        if not self.tables[table]:
            return {"error": f"Table is empty: {table}"}
        
        first_row = self.tables[table][0]
        schema = {
            "columns": list(first_row.keys()),
            "types": {k: type(v).__name__ for k, v in first_row.items()}
        }
        
        return schema
    
    def _log_query(self, table: str, column: str, value: str, status: str, details: str):
        """Audit logging"""
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "table": table,
            "where_column": column,
            "where_value": value,
            "status": status,
            "details": details
        })
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Override to add database tools"""
        if tool_name == "query_table":
            return self.query_table(
                arguments.get("table"),
                arguments.get("where_column"),
                arguments.get("where_value"),
                arguments.get("limit", 10)
            )
        elif tool_name == "get_table_schema":
            return self.get_table_schema(arguments.get("table"))
        else:
            return super().call_tool(tool_name, arguments)

# Create database server
db_server = DatabaseMCPServer()

print("✓ Database MCP Server created!")
print(f"Tables: {list(db_server.tables.keys())}")

# Test queries
print("\n" + "="*60)
print("TEST 1: Get schema of employees table")
print("="*60)
response = db_server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "get_table_schema",
        "arguments": {"table": "employees"}
    }
})
print(json.dumps(response["result"], indent=2))

print("\n" + "="*60)
print("TEST 2: Query employees in Engineering department")
print("="*60)
response = db_server.handle_request({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "query_table",
        "arguments": {
            "table": "employees",
            "where_column": "department",
            "where_value": "Engineering"
        }
    }
})
print(json.dumps(response["result"], indent=2))

print("\n" + "="*60)
print("AUDIT LOG:")
print("="*60)
for entry in db_server.audit_log:
    print(f"  {entry['timestamp']} | {entry['table']} | {entry['status']}")


---

# Section 7: Add Authentication, Secrets & Permission Scopes

Security in MCP servers requires:
- ✅ Token-based authentication
- ✅ Secret management (.env files)
- ✅ Role-based access control (RBAC)
- ✅ Audit logging


In [ ]:
# Secure MCP Server with Authentication
class SecureMCPServer(MinimalMCPServer):
    """
    MCP Server with:
    - Token-based authentication
    - Role-based access control (RBAC)
    - Scope-based permissions
    """
    
    def __init__(self, secret_key: str = "dev-secret-key-change-in-prod"):
        super().__init__("secure-server")
        self.secret_key = secret_key
        
        # User/token database
        self.tokens = {
            "token-admin-001": {
                "user": "admin",
                "role": "admin",
                "scopes": ["read", "write", "delete", "admin"]
            },
            "token-user-001": {
                "user": "user1",
                "role": "user",
                "scopes": ["read"]
            },
            "token-editor-001": {
                "user": "editor1",
                "role": "editor",
                "scopes": ["read", "write"]
            }
        }
        
        # Tool permissions map: tool_name -> required_scope
        self.tool_permissions = {
            "read_data": "read",
            "write_data": "write",
            "delete_data": "delete"
        }
    
    def verify_token(self, token: str) -> Optional[Dict]:
        """Verify and decode token"""
        if token not in self.tokens:
            return None
        return self.tokens[token]
    
    def check_scope(self, token_info: Dict, required_scope: str) -> bool:
        """Check if token has required scope"""
        return required_scope in token_info.get("scopes", [])
    
    def handle_request_authenticated(self, request: Dict[str, Any], token: str):
        """Handle request with authentication"""
        # Verify token
        token_info = self.verify_token(token)
        if not token_info:
            return {
                "jsonrpc": "2.0",
                "id": request.get("id"),
                "error": {"code": 401, "message": "Invalid or missing token"}
            }
        
        # Check tool permissions
        tool_name = request.get("params", {}).get("name")
        if tool_name in self.tool_permissions:
            required_scope = self.tool_permissions[tool_name]
            if not self.check_scope(token_info, required_scope):
                return {
                    "jsonrpc": "2.0",
                    "id": request.get("id"),
                    "error": {
                        "code": 403,
                        "message": f"Insufficient permissions. Required: {required_scope}"
                    }
                }
        
        # Request is authorized, process normally
        return self.handle_request(request)

# Create secure server
secure_server = SecureMCPServer()

print("✓ Secure MCP Server created!")
print("Available tokens:")
for token, info in secure_server.tokens.items():
    print(f"  {token}: {info['user']} ({info['role']}) - scopes: {info['scopes']}")

# Test 1: Valid token with permission
print("\n" + "="*60)
print("TEST 1: Admin token calling read_data (ALLOWED)")
print("="*60)
response = secure_server.handle_request_authenticated({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "add",
        "arguments": {"a": 5, "b": 3}
    }
}, "token-admin-001")
print(json.dumps(response, indent=2))

# Test 2: User token insufficient permissions
print("\n" + "="*60)
print("TEST 2: User token (read-only) trying delete operation")
print("="*60)
response = secure_server.handle_request_authenticated({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "delete_data",
        "arguments": {"id": 123}
    }
}, "token-user-001")
print(json.dumps(response, indent=2))

# Test 3: Invalid token
print("\n" + "="*60)
print("TEST 3: Invalid token (NOT ALLOWED)")
print("="*60)
response = secure_server.handle_request_authenticated({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "add",
        "arguments": {"a": 1, "b": 2}
    }
}, "invalid-token-xyz")
print(json.dumps(response, indent=2))


---

# Section 8: Connect Agent to Multiple MCP Servers (Orchestration)

An intelligent agent should:
1. Discover capabilities from all connected servers
2. Route requests to the appropriate server
3. Handle failures gracefully with retries/fallbacks
4. Aggregate results from multiple sources


In [ ]:
# Multi-Server Agent Orchestrator
class MultiServerAgent:
    """
    Agent that orchestrates multiple MCP servers with:
    - Capability discovery
    - Intelligent routing
    - Fallback handling
    - Result aggregation
    """
    
    def __init__(self, name: str):
        self.name = name
        self.servers = {}
        self.server_capabilities = {}
    
    def register_server(self, server_name: str, server: MinimalMCPServer, token: str = None):
        """Register an MCP server"""
        self.servers[server_name] = {
            "instance": server,
            "token": token
        }
        
        # Discover capabilities
        self._discover_capabilities(server_name, server)
        
        print(f"  ✓ Registered: {server_name}")
    
    def _discover_capabilities(self, server_name: str, server: MinimalMCPServer):
        """Discover what tools and resources are available"""
        self.server_capabilities[server_name] = {
            "tools": [],
            "resources": []
        }
        
        # Get tools
        tools_response = server.handle_request({
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/list"
        })
        if "tools" in tools_response.get("result", {}):
            for tool in tools_response["result"]["tools"]:
                self.server_capabilities[server_name]["tools"].append(tool["name"])
        
        # Get resources
        resources_response = server.handle_request({
            "jsonrpc": "2.0",
            "id": 2,
            "method": "resources/list"
        })
        if "resources" in resources_response.get("result", {}):
            for resource in resources_response["result"]["resources"]:
                self.server_capabilities[server_name]["resources"].append(resource["uri"])
    
    def find_tool(self, tool_name: str) -> Optional[str]:
        """Find which server has a specific tool"""
        for server_name, capabilities in self.server_capabilities.items():
            if tool_name in capabilities["tools"]:
                return server_name
        return None
    
    def find_servers_with_resource(self, resource_uri: str) -> List[str]:
        """Find all servers with a specific resource"""
        matching_servers = []
        for server_name, capabilities in self.server_capabilities.items():
            if resource_uri in capabilities["resources"]:
                matching_servers.append(server_name)
        return matching_servers
    
    def call_tool_with_fallback(self, tool_name: str, arguments: Dict):
        """Call tool with fallback servers"""
        server_name = self.find_tool(tool_name)
        
        if not server_name:
            return {"error": f"Tool '{tool_name}' not found in any server"}
        
        try:
            # Try primary server
            server = self.servers[server_name]["instance"]
            response = server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "tools/call",
                "params": {
                    "name": tool_name,
                    "arguments": arguments
                }
            })
            return {
                "status": "success",
                "server": server_name,
                "result": response.get("result")
            }
        except Exception as e:
            return {
                "status": "error",
                "server": server_name,
                "error": str(e)
            }
    
    def query_resources(self, resource_pattern: str):
        """Query multiple resources across servers"""
        results = {}
        
        # Find servers with matching resources
        matching_servers = self.find_servers_with_resource(resource_pattern)
        
        for server_name in matching_servers:
            server = self.servers[server_name]["instance"]
            response = server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "resources/read",
                "params": {"uri": resource_pattern}
            })
            results[server_name] = response.get("result")
        
        return results
    
    def print_capabilities(self):
        """Print discovered capabilities"""
        print(f"\n🤖 Agent: {self.name}")
        print("="*60)
        for server_name, capabilities in self.server_capabilities.items():
            print(f"\n📦 Server: {server_name}")
            print(f"   Tools: {', '.join(capabilities['tools']) or 'None'}")
            print(f"   Resources: {', '.join(capabilities['resources']) or 'None'}")

# Create multi-server agent
orchestrator = MultiServerAgent("Enterprise Agent")

print("Registering servers...")
orchestrator.register_server("math-server", server)
orchestrator.register_server("rag-server", rag_server)
orchestrator.register_server("db-server", db_server)

orchestrator.print_capabilities()

# Test 1: Find and call tool
print("\n" + "="*60)
print("TEST: Agent finding and calling 'search_documents' tool")
print("="*60)
result = orchestrator.call_tool_with_fallback("search_documents", {"query": "DevOps"})
print(json.dumps(result, indent=2))

# Test 2: Find tool that doesn't exist
print("\n" + "="*60)
print("TEST: Agent looking for non-existent tool")
print("="*60)
result = orchestrator.call_tool_with_fallback("generate_report", {})
print(json.dumps(result, indent=2))

# Test 3: Query resources across servers
print("\n" + "="*60)
print("TEST: Agent querying server info from all servers")
print("="*60)
results = orchestrator.query_resources("server://info")
for server, data in results.items():
    print(f"\n{server}: {data}")


---

# Section 9: Using Pre-built MCP Servers - Azure DevOps (ADO)

Pre-built MCP servers are ready-to-use integrations for popular platforms.

**Azure DevOps MCP Server** exposes:
- Work Items (user stories, bugs, tasks)
- Pipelines (CI/CD)
- Repositories (Git)
- Boards and iterations

Let's create a simulated ADO MCP server:


In [ ]:
# Simulated Azure DevOps MCP Server
class AzureDevOpsMCPServer(MinimalMCPServer):
    """
    Simulated pre-built MCP server for Azure DevOps
    In production, this would connect to real Azure DevOps APIs
    """
    
    def __init__(self):
        super().__init__("azure-devops-server")
        
        # Simulated work items
        self.work_items = [
            {
                "id": "WI-1001",
                "title": "Implement user authentication",
                "type": "User Story",
                "state": "Active",
                "priority": "High",
                "assignee": "Alice",
                "sprint": "Sprint 24"
            },
            {
                "id": "WI-1002",
                "title": "Fix login bug on mobile",
                "type": "Bug",
                "state": "Active",
                "priority": "High",
                "assignee": "Bob",
                "sprint": "Sprint 24"
            },
            {
                "id": "WI-1003",
                "title": "Update documentation",
                "type": "Task",
                "state": "Closed",
                "priority": "Low",
                "assignee": "Charlie",
                "sprint": "Sprint 23"
            }
        ]
        
        # Simulated pipelines
        self.pipelines = [
            {
                "id": "Pipeline-001",
                "name": "Main CI/CD",
                "status": "succeeded",
                "branch": "main",
                "commit": "abc123def456",
                "timestamp": "2024-01-15T10:30:00Z"
            },
            {
                "id": "Pipeline-002",
                "name": "PR Check",
                "status": "inProgress",
                "branch": "feature/auth",
                "commit": "xyz789uvw012",
                "timestamp": "2024-01-15T14:15:00Z"
            }
        ]
        
        # Add tools
        self.tools.update({
            "list_work_items": {
                "description": "List work items from Azure DevOps",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "state": {"type": "string", "enum": ["Active", "Closed", "all"]},
                        "type": {"type": "string", "enum": ["User Story", "Bug", "Task", "all"]}
                    }
                }
            },
            "get_work_item": {
                "description": "Get details of a specific work item",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "work_item_id": {"type": "string"}
                    },
                    "required": ["work_item_id"]
                }
            },
            "list_pipelines": {
                "description": "List CI/CD pipelines",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "status": {"type": "string", "enum": ["succeeded", "failed", "inProgress", "all"]}
                    }
                }
            },
            "create_work_item": {
                "description": "Create a new work item",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "title": {"type": "string"},
                        "type": {"type": "string", "enum": ["User Story", "Bug", "Task"]},
                        "priority": {"type": "string", "enum": ["Low", "Medium", "High"]}
                    },
                    "required": ["title", "type"]
                }
            }
        })
    
    def list_work_items(self, state: str = "all", type_filter: str = "all"):
        """List work items with optional filters"""
        items = self.work_items
        
        if state != "all":
            items = [w for w in items if w["state"] == state]
        
        if type_filter != "all":
            items = [w for w in items if w["type"] == type_filter]
        
        return {
            "total": len(items),
            "work_items": items
        }
    
    def get_work_item(self, work_item_id: str):
        """Get specific work item"""
        for item in self.work_items:
            if item["id"] == work_item_id:
                return {"work_item": item}
        return {"error": f"Work item not found: {work_item_id}"}
    
    def list_pipelines(self, status: str = "all"):
        """List pipelines with optional status filter"""
        pipelines = self.pipelines
        
        if status != "all":
            pipelines = [p for p in pipelines if p["status"] == status]
        
        return {
            "total": len(pipelines),
            "pipelines": pipelines
        }
    
    def create_work_item(self, title: str, type_: str, priority: str = "Medium"):
        """Create new work item"""
        new_id = f"WI-{len(self.work_items) + 1001}"
        new_item = {
            "id": new_id,
            "title": title,
            "type": type_,
            "state": "New",
            "priority": priority,
            "assignee": None,
            "sprint": None
        }
        self.work_items.append(new_item)
        return {"created": new_item}
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Override to add ADO-specific tools"""
        if tool_name == "list_work_items":
            return self.list_work_items(
                arguments.get("state", "all"),
                arguments.get("type", "all")
            )
        elif tool_name == "get_work_item":
            return self.get_work_item(arguments.get("work_item_id"))
        elif tool_name == "list_pipelines":
            return self.list_pipelines(arguments.get("status", "all"))
        elif tool_name == "create_work_item":
            return self.create_work_item(
                arguments.get("title"),
                arguments.get("type"),
                arguments.get("priority", "Medium")
            )
        else:
            return super().call_tool(tool_name, arguments)

# Create ADO server
ado_server = AzureDevOpsMCPServer()

print("✓ Azure DevOps MCP Server created!")

# Test 1: List active work items
print("\n" + "="*60)
print("TEST 1: List active work items")
print("="*60)
response = ado_server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "list_work_items",
        "arguments": {"state": "Active"}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 2: List pipelines
print("\n" + "="*60)
print("TEST 2: List all pipelines")
print("="*60)
response = ado_server.handle_request({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "list_pipelines",
        "arguments": {"status": "all"}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 3: Create work item
print("\n" + "="*60)
print("TEST 3: Create new work item")
print("="*60)
response = ado_server.handle_request({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "create_work_item",
        "arguments": {
            "title": "Implement API rate limiting",
            "type": "User Story",
            "priority": "High"
        }
    }
})
print(json.dumps(response["result"], indent=2))


---

# Section 10: Using Pre-built MCP Servers - Microsoft Azure

**Microsoft Azure MCP Server** exposes:
- Resource Groups and resources
- Virtual Machines
- App Services and Functions
- Storage and Databases
- Application Insights (logs, metrics)

Let's create a simulated Azure MCP server:


In [ ]:
# Simulated Microsoft Azure MCP Server
class MicrosoftAzureMCPServer(MinimalMCPServer):
    """
    Simulated pre-built MCP server for Microsoft Azure
    """
    
    def __init__(self):
        super().__init__("microsoft-azure-server")
        
        # Simulated resources
        self.resource_groups = [
            {
                "name": "rg-prod-us-east",
                "location": "eastus",
                "resources": 8,
                "status": "active"
            },
            {
                "name": "rg-dev-us-west",
                "location": "westus",
                "resources": 3,
                "status": "active"
            }
        ]
        
        # Simulated app services
        self.app_services = [
            {
                "name": "api-server-prod",
                "rg": "rg-prod-us-east",
                "status": "running",
                "instances": 3,
                "tier": "Premium",
                "cpu": 45.2,
                "memory": 67.8
            },
            {
                "name": "web-app-prod",
                "rg": "rg-prod-us-east",
                "status": "running",
                "instances": 2,
                "tier": "Standard",
                "cpu": 23.1,
                "memory": 42.3
            }
        ]
        
        # Simulated metrics
        self.metrics = {
            "api-server-prod": {
                "requests": 15420,
                "errors": 12,
                "latency_ms": 234,
                "cpu_percent": 45.2
            }
        }
        
        # Add tools
        self.tools.update({
            "list_resource_groups": {
                "description": "List Azure resource groups",
                "input_schema": {"type": "object"}
            },
            "list_app_services": {
                "description": "List App Services in a resource group",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "resource_group": {"type": "string"}
                    }
                }
            },
            "get_app_metrics": {
                "description": "Get performance metrics for an app service",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "app_name": {"type": "string"}
                    },
                    "required": ["app_name"]
                }
            },
            "scale_app_service": {
                "description": "Scale an app service (change number of instances)",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "app_name": {"type": "string"},
                        "instances": {"type": "integer", "minimum": 1, "maximum": 10}
                    },
                    "required": ["app_name", "instances"]
                }
            }
        })
    
    def list_resource_groups(self):
        """List all resource groups"""
        return {
            "total": len(self.resource_groups),
            "resource_groups": self.resource_groups
        }
    
    def list_app_services(self, resource_group: str = None):
        """List app services"""
        apps = self.app_services
        
        if resource_group:
            apps = [a for a in apps if a["rg"] == resource_group]
        
        return {
            "total": len(apps),
            "app_services": apps
        }
    
    def get_app_metrics(self, app_name: str):
        """Get app metrics"""
        if app_name in self.metrics:
            return {
                "app": app_name,
                "metrics": self.metrics[app_name],
                "timestamp": datetime.now().isoformat()
            }
        return {"error": f"App not found: {app_name}"}
    
    def scale_app_service(self, app_name: str, instances: int):
        """Scale an app service"""
        for app in self.app_services:
            if app["name"] == app_name:
                old_instances = app["instances"]
                app["instances"] = instances
                return {
                    "success": True,
                    "app": app_name,
                    "old_instances": old_instances,
                    "new_instances": instances,
                    "message": f"Scaled {app_name} from {old_instances} to {instances} instances"
                }
        return {"error": f"App not found: {app_name}"}
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Override to add Azure-specific tools"""
        if tool_name == "list_resource_groups":
            return self.list_resource_groups()
        elif tool_name == "list_app_services":
            return self.list_app_services(arguments.get("resource_group"))
        elif tool_name == "get_app_metrics":
            return self.get_app_metrics(arguments.get("app_name"))
        elif tool_name == "scale_app_service":
            return self.scale_app_service(
                arguments.get("app_name"),
                arguments.get("instances")
            )
        else:
            return super().call_tool(tool_name, arguments)

# Create Azure server
azure_server = MicrosoftAzureMCPServer()

print("✓ Microsoft Azure MCP Server created!")

# Test 1: List resource groups
print("\n" + "="*60)
print("TEST 1: List Azure resource groups")
print("="*60)
response = azure_server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "list_resource_groups",
        "arguments": {}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 2: Get app metrics
print("\n" + "="*60)
print("TEST 2: Get performance metrics")
print("="*60)
response = azure_server.handle_request({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "get_app_metrics",
        "arguments": {"app_name": "api-server-prod"}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 3: Scale app service
print("\n" + "="*60)
print("TEST 3: Auto-scale app service (CPU overloaded)")
print("="*60)
response = azure_server.handle_request({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "scale_app_service",
        "arguments": {"app_name": "api-server-prod", "instances": 5}
    }
})
print(json.dumps(response["result"], indent=2))


---

# Section 11: Using Pre-built MCP Servers - SonarQube

**SonarQube MCP Server** exposes:
- Code quality metrics
- Security vulnerabilities
- Code coverage data
- Technical debt
- Code smells

Let's create a simulated SonarQube MCP server:


In [ ]:
# Simulated SonarQube MCP Server
class SonarQubeMCPServer(MinimalMCPServer):
    """
    Simulated pre-built MCP server for SonarQube code quality
    """
    
    def __init__(self):
        super().__init__("sonarqube-server")
        
        # Simulated projects
        self.projects = [
            {
                "key": "com.myapp:backend",
                "name": "Backend API",
                "quality_gate": "PASSED",
                "rating": "A",
                "bugs": 3,
                "vulnerabilities": 1,
                "code_smells": 25,
                "coverage": 82.5,
                "debt_days": 2.3
            },
            {
                "key": "com.myapp:frontend",
                "name": "Frontend Web",
                "quality_gate": "PASSED",
                "rating": "B",
                "bugs": 8,
                "vulnerabilities": 2,
                "code_smells": 45,
                "coverage": 65.0,
                "debt_days": 5.1
            }
        ]
        
        # Simulated issues
        self.issues = [
            {
                "key": "issue-001",
                "project": "com.myapp:backend",
                "type": "BUG",
                "severity": "HIGH",
                "message": "Potential null pointer exception",
                "file": "src/auth.py",
                "line": 42
            },
            {
                "key": "issue-002",
                "project": "com.myapp:backend",
                "type": "VULNERABILITY",
                "severity": "CRITICAL",
                "message": "SQL injection risk detected",
                "file": "src/database.py",
                "line": 87
            },
            {
                "key": "issue-003",
                "project": "com.myapp:frontend",
                "type": "CODE_SMELL",
                "severity": "MINOR",
                "message": "Function too complex",
                "file": "src/components/Dashboard.js",
                "line": 156
            }
        ]
        
        # Add tools
        self.tools.update({
            "list_projects": {
                "description": "List all projects in SonarQube",
                "input_schema": {"type": "object"}
            },
            "get_project_metrics": {
                "description": "Get quality metrics for a project",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "project_key": {"type": "string"}
                    },
                    "required": ["project_key"]
                }
            },
            "list_issues": {
                "description": "List issues in a project",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "project_key": {"type": "string"},
                        "severity": {"type": "string", "enum": ["CRITICAL", "HIGH", "MEDIUM", "LOW", "all"]}
                    },
                    "required": ["project_key"]
                }
            },
            "get_quality_gate": {
                "description": "Check if project passes quality gate",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "project_key": {"type": "string"}
                    },
                    "required": ["project_key"]
                }
            }
        })
    
    def list_projects(self):
        """List all projects"""
        return {
            "total": len(self.projects),
            "projects": self.projects
        }
    
    def get_project_metrics(self, project_key: str):
        """Get detailed metrics for a project"""
        for project in self.projects:
            if project["key"] == project_key:
                return {
                    "project": project_key,
                    "metrics": project
                }
        return {"error": f"Project not found: {project_key}"}
    
    def list_issues(self, project_key: str, severity: str = "all"):
        """List issues in a project"""
        issues = [i for i in self.issues if i["project"] == project_key]
        
        if severity != "all":
            issues = [i for i in issues if i["severity"] == severity]
        
        return {
            "project": project_key,
            "total": len(issues),
            "issues": issues
        }
    
    def get_quality_gate(self, project_key: str):
        """Check quality gate status"""
        for project in self.projects:
            if project["key"] == project_key:
                status = project["quality_gate"]
                return {
                    "project": project_key,
                    "status": status,
                    "passed": status == "PASSED",
                    "rating": project["rating"],
                    "recommendations": self._get_recommendations(project)
                }
        return {"error": f"Project not found: {project_key}"}
    
    def _get_recommendations(self, project):
        """Generate recommendations based on metrics"""
        recommendations = []
        
        if project["bugs"] > 5:
            recommendations.append(f"Reduce bugs ({project['bugs']} found)")
        if project["vulnerabilities"] > 0:
            recommendations.append(f"Fix vulnerabilities ({project['vulnerabilities']} found)")
        if project["coverage"] < 70:
            recommendations.append(f"Increase test coverage (currently {project['coverage']}%)")
        if project["debt_days"] > 3:
            recommendations.append(f"Address technical debt ({project['debt_days']} days)")
        
        return recommendations
    
    def call_tool(self, tool_name: str, arguments: Dict[str, Any]):
        """Override to add SonarQube-specific tools"""
        if tool_name == "list_projects":
            return self.list_projects()
        elif tool_name == "get_project_metrics":
            return self.get_project_metrics(arguments.get("project_key"))
        elif tool_name == "list_issues":
            return self.list_issues(
                arguments.get("project_key"),
                arguments.get("severity", "all")
            )
        elif tool_name == "get_quality_gate":
            return self.get_quality_gate(arguments.get("project_key"))
        else:
            return super().call_tool(tool_name, arguments)

# Create SonarQube server
sonarqube_server = SonarQubeMCPServer()

print("✓ SonarQube MCP Server created!")

# Test 1: List projects
print("\n" + "="*60)
print("TEST 1: List SonarQube projects")
print("="*60)
response = sonarqube_server.handle_request({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "list_projects",
        "arguments": {}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 2: Get project metrics
print("\n" + "="*60)
print("TEST 2: Get project quality metrics")
print("="*60)
response = sonarqube_server.handle_request({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "get_project_metrics",
        "arguments": {"project_key": "com.myapp:backend"}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 3: Get quality gate with recommendations
print("\n" + "="*60)
print("TEST 3: Check quality gate + recommendations")
print("="*60)
response = sonarqube_server.handle_request({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "get_quality_gate",
        "arguments": {"project_key": "com.myapp:frontend"}
    }
})
print(json.dumps(response["result"], indent=2))

# Test 4: List critical issues
print("\n" + "="*60)
print("TEST 4: List CRITICAL severity issues")
print("="*60)
response = sonarqube_server.handle_request({
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "list_issues",
        "arguments": {
            "project_key": "com.myapp:backend",
            "severity": "CRITICAL"
        }
    }
})
print(json.dumps(response["result"], indent=2))


---

# Section 12: Compose Multi-Server Workflows (Enterprise Agent)

Now let's build an enterprise agent that:
1. Uses Azure DevOps to track work
2. Uses SonarQube for code quality
3. Uses Azure for infrastructure
4. Orchestrates complex workflows


In [ ]:
# Enterprise Workflow: Code Quality To Production
class EnterpriseWorkflow:
    """
    Multi-server workflow:
    1. Check code quality in SonarQube
    2. If issues found, create work item in ADO
    3. If app under load, scale in Azure
    4. Track everything
    """
    
    def __init__(self, orchestrator: MultiServerAgent):
        self.orchestrator = orchestrator
        self.execution_log = []
    
    def log(self, stage: str, message: str):
        """Log workflow execution"""
        self.execution_log.append({
            "timestamp": datetime.now().isoformat(),
            "stage": stage,
            "message": message
        })
        print(f"  [{stage}] {message}")
    
    def check_and_respond_to_quality_issues(self, project_key: str):
        """
        WORKFLOW STEP 1: Check SonarQube quality, create ADO work item if issues
        """
        print("\n" + "="*60)
        print("🔄 WORKFLOW: Check Code Quality & Create Work Items")
        print("="*60)
        
        self.log("QA_CHECK", f"Checking SonarQube project: {project_key}")
        
        # Step 1: Get quality metrics from SonarQube
        quality_result = self.orchestrator.call_tool_with_fallback(
            "get_quality_gate",
            {"project_key": project_key}
        )
        
        if quality_result["status"] != "success":
            self.log("ERROR", f"Failed to get quality metrics: {quality_result}")
            return
        
        quality_data = quality_result["result"]
        self.log("QUALITY_RESULT", f"Rating: {quality_data['rating']}, Passed: {quality_data['passed']}")
        
        # Step 2: If quality issues exist, create work items
        if not quality_data["passed"] or quality_data["rating"] in ["C", "D", "E"]:
            recommendations = quality_data.get("recommendations", [])
            
            for i, rec in enumerate(recommendations):
                self.log("CREATE_WORKITEM", f"Creating work item for: {rec}")
                
                # Create work item in ADO
                ado_result = self.orchestrator.call_tool_with_fallback(
                    "create_work_item",
                    {
                        "title": f"[Code Quality] {rec}",
                        "type": "Task",
                        "priority": "High"
                    }
                )
                
                if ado_result["status"] == "success":
                    self.log("SUCCESS", f"Work item created: {ado_result['result']['created']['id']}")
        else:
            self.log("SUCCESS", "Code quality is excellent! No work items needed.")
    
    def monitor_and_scale(self, app_name: str):
        """
        WORKFLOW STEP 2: Check app metrics, auto-scale if needed
        """
        print("\n" + "="*60)
        print("🔄 WORKFLOW: Monitor Performance & Auto-Scale")
        print("="*60)
        
        self.log("MONITORING", f"Checking metrics for: {app_name}")
        
        # Get metrics from Azure
        metrics_result = self.orchestrator.call_tool_with_fallback(
            "get_app_metrics",
            {"app_name": app_name}
        )
        
        if metrics_result["status"] != "success":
            self.log("ERROR", f"Failed to get metrics: {metrics_result}")
            return
        
        metrics = metrics_result["result"]["metrics"]
        cpu = metrics.get("cpu_percent", 0)
        
        self.log("METRICS", f"CPU: {cpu}%, Latency: {metrics.get('latency_ms')}ms")
        
        # Auto-scale if CPU high
        if cpu > 70:
            self.log("ALERT", f"CPU too high ({cpu}%)! Scaling up...")
            
            scale_result = self.orchestrator.call_tool_with_fallback(
                "scale_app_service",
                {"app_name": app_name, "instances": 5}
            )
            
            if scale_result["status"] == "success":
                msg = scale_result["result"].get("message", "Scaled")
                self.log("SUCCESS", msg)
        else:
            self.log("OK", f"CPU normal ({cpu}%)")
    
    def full_pipeline(self, project_key: str, app_name: str):
        """Run complete workflow"""
        print("\n" + "🚀 ENTERPRISE WORKFLOW STARTING" + "\n")
        
        self.check_and_respond_to_quality_issues(project_key)
        self.monitor_and_scale(app_name)
        
        print("\n" + "="*60)
        print("📋 WORKFLOW EXECUTION LOG:")
        print("="*60)
        for entry in self.execution_log:
            print(f"  {entry['timestamp']} | {entry['stage']:12} | {entry['message']}")

# Register all servers with orchestrator
print("📦 Registering all enterprise servers...")
orchestrator.register_server("ado", ado_server)
orchestrator.register_server("azure", azure_server)
orchestrator.register_server("sonarqube", sonarqube_server)

# Create and run enterprise workflow
workflow = EnterpriseWorkflow(orchestrator)
workflow.full_pipeline("com.myapp:frontend", "api-server-prod")

print("\n✓ Enterprise workflow completed!")


---

# Section 13: Containerize & Deploy Your MCP Server

### Deployment Options:
1. **Docker** - Containerized local/cloud deployment
2. **Azure Container Instances** - Serverless containers
3. **Azure App Service** - Managed web apps
4. **Kubernetes** - Advanced orchestration

Let's create deployment artifacts:


In [ ]:
# Generate Dockerfile for MCP Server
dockerfile_content = """FROM python:3.11-slim

WORKDIR /app

# Copy requirements
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy server code
COPY mcp_server.py .

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD python -c "import requests; requests.get('http://localhost:8000/health')" || exit 1

# Expose port
EXPOSE 8000

# Run server
CMD ["python", "mcp_server.py"]
"""

print("📄 Dockerfile for MCP Server:")
print("="*60)
print(dockerfile_content)

# Generate Docker Compose for local development
docker_compose_content = """version: '3.8'

services:
  mcp-math-server:
    build: .
    container_name: mcp-math-server
    ports:
      - "8001:8000"
    environment:
      - MCP_SERVER_NAME=math-server
      - DEBUG=true
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 10s
      timeout: 5s
      retries: 3

  mcp-db-server:
    build: .
    container_name: mcp-db-server
    ports:
      - "8002:8000"
    environment:
      - MCP_SERVER_NAME=database-server
      - DEBUG=true
    depends_on:
      - mcp-math-server

networks:
  mcp-net:
    driver: bridge
"""

print("\n📄 Docker Compose (Local Development):")
print("="*60)
print(docker_compose_content)

# Generate requirements.txt
requirements_txt = """pydantic>=2.0
json-rpc>=1.14.1
httpx>=0.24.0
aiofiles>=23.0.0
python-dotenv>=1.0.0
flask>=2.3.0
gunicorn>=21.0.0
pymongo>=4.5.0
sqlalchemy>=2.0.0
pandas>=2.0.0
requests>=2.31.0
"""

print("\n📄 requirements.txt:")
print("="*60)
print(requirements_txt)

print("\n✓ Deployment artifacts generated!")


---

# Section 14: Testing, Logging & Troubleshooting

Comprehensive testing and diagnostics for MCP servers:
- ✅ Unit tests for tools
- ✅ Integration tests for multi-server flows
- ✅ Structured logging
- ✅ Common error diagnostics


In [ ]:
# Comprehensive Testing & Diagnostics
class MCPServerTester:
    """
    Test suite for MCP servers with:
    - Tool contract validation
    - Error handling tests
    - Performance metrics
    - Structured logging
    """
    
    def __init__(self, server: MinimalMCPServer):
        self.server = server
        self.tests_run = 0
        self.tests_passed = 0
        self.tests_failed = 0
        self.logs = []
    
    def log(self, level: str, test: str, message: str):
        """Structured logging"""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "level": level,
            "test": test,
            "message": message
        }
        self.logs.append(entry)
        emoji = "✓" if level == "PASS" else "✗" if level == "FAIL" else "ℹ"
        print(f"  {emoji} [{level:4}] {test:30} {message}")
    
    def test_tool_exists(self, tool_name: str) -> bool:
        """Test if tool is registered"""
        self.tests_run += 1
        exists = tool_name in self.server.tools
        
        if exists:
            self.tests_passed += 1
            self.log("PASS", f"Tool exists: {tool_name}", "")
        else:
            self.tests_failed += 1
            self.log("FAIL", f"Tool exists: {tool_name}", "Tool not found")
        
        return exists
    
    def test_tool_execution(self, tool_name: str, arguments: Dict) -> bool:
        """Test tool execution"""
        self.tests_run += 1
        
        try:
            response = self.server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "tools/call",
                "params": {
                    "name": tool_name,
                    "arguments": arguments
                }
            })
            
            if "error" not in response.get("result", {}):
                self.tests_passed += 1
                self.log("PASS", f"Tool execution: {tool_name}", "")
                return True
            else:
                self.tests_failed += 1
                error = response["result"].get("error", "Unknown error")
                self.log("FAIL", f"Tool execution: {tool_name}", str(error))
                return False
        
        except Exception as e:
            self.tests_failed += 1
            self.log("FAIL", f"Tool execution: {tool_name}", f"Exception: {str(e)[:50]}")
            return False
    
    def test_invalid_tool(self) -> bool:
        """Test error handling for invalid tool"""
        self.tests_run += 1
        
        try:
            response = self.server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "tools/call",
                "params": {
                    "name": "nonexistent_tool",
                    "arguments": {}
                }
            })
            
            # Should return error
            if "error" in response.get("result", {}) or "error" in response:
                self.tests_passed += 1
                self.log("PASS", "Error handling: invalid tool", "Correctly rejected")
                return True
            else:
                self.tests_failed += 1
                self.log("FAIL", "Error handling: invalid tool", "Should reject invalid tool")
                return False
        
        except Exception as e:
            self.tests_passed += 1
            self.log("PASS", "Error handling: invalid tool", "Exception raised (OK)")
            return True
    
    def test_resources(self) -> bool:
        """Test resource discovery"""
        self.tests_run += 1
        
        try:
            response = self.server.handle_request({
                "jsonrpc": "2.0",
                "id": 1,
                "method": "resources/list"
            })
            
            if "resources" in response.get("result", {}):
                count = len(response["result"]["resources"])
                self.tests_passed += 1
                self.log("PASS", "Resource discovery", f"Found {count} resources")
                return True
            else:
                self.tests_failed += 1
                self.log("FAIL", "Resource discovery", "No resources returned")
                return False
        
        except Exception as e:
            self.tests_failed += 1
            self.log("FAIL", "Resource discovery", str(e)[:50])
            return False
    
    def run_full_test_suite(self) -> Dict:
        """Run all tests"""
        print(f"\n{'='*60}")
        print(f"🧪 Testing MCP Server: {self.server.name}")
        print(f"{'='*60}\n")
        
        # Test basic functionality
        for tool_name in list(self.server.tools.keys())[:3]:  # Test first 3 tools
            self.test_tool_exists(tool_name)
        
        # Test invalid tool
        self.test_invalid_tool()
        
        # Test resources
        self.test_resources()
        
        # Print summary
        print(f"\n{'='*60}")
        print(f"TEST SUMMARY")
        print(f"{'='*60}")
        print(f"  Total Tests: {self.tests_run}")
        print(f"  Passed: {self.tests_passed} ✓")
        print(f"  Failed: {self.tests_failed} ✗")
        print(f"  Success Rate: {100*self.tests_passed//max(self.tests_run, 1)}%")
        
        return {
            "total": self.tests_run,
            "passed": self.tests_passed,
            "failed": self.tests_failed,
            "success_rate": 100*self.tests_passed//max(self.tests_run, 1)
        }

# Run tests on all servers
print("🧪 MCP SERVER TEST SUITE\n")

testers = {
    "Math Server": MCPServerTester(server),
    "RAG Server": MCPServerTester(rag_server),
    "Database Server": MCPServerTester(db_server),
    "Azure DevOps": MCPServerTester(ado_server)
}

results = {}
for name, tester in testers.items():
    results[name] = tester.run_full_test_suite()
    print()

# Overall summary
print("\n" + "="*60)
print("📊 OVERALL TEST RESULTS")
print("="*60)
for name, result in results.items():
    status = "✓ PASS" if result["success_rate"] >= 80 else "⚠ CHECK"
    print(f"{status} | {name:20} | {result['success_rate']}% success rate")


---

# Common MCP Issues & Troubleshooting

### Issue 1: "Tool not found"
**Cause**: Tool not registered in server
**Fix**: Ensure tool is in `self.tools` dictionary

### Issue 2: "Invalid token / Authentication failed"
**Cause**: Invalid or missing authentication token
**Fix**: Check token format, ensure proper RBAC scopes

### Issue 3: "JSON-RPC parsing error"
**Cause**: Malformed request JSON
**Fix**: Validate request format matches JSON-RPC 2.0 spec

### Issue 4: "Timeout waiting for response"
**Cause**: Long-running tool, network lag
**Fix**: Add timeout handling, use async/await

### Issue 5: "Connection refused"
**Cause**: MCP server not running or wrong port
**Fix**: Verify server is running, check firewall/port settings


---

# Summary: Your MCP Learning Journey

## What You've Learned

✅ **MCP Fundamentals**: Protocol, architecture, components
✅ **Build Custom Servers**: Tools, resources, authentication
✅ **Real-World Patterns**: RAG, database access, multi-server orchestration
✅ **Pre-built Integrations**: Azure DevOps, Microsoft Azure, SonarQube
✅ **Agent Development**: Autonomous systems using MCP
✅ **Deployment**: Docker, containerization, cloud hosting
✅ **Testing & Debugging**: Comprehensive test suites, troubleshooting

## Next Steps

### Beginner → Intermediate (1-2 weeks)
1. Set up local MCP servers
2. Build your first custom server (document Q&A or database access)
3. Create simple 2-server agent workflow
4. Deploy locally with Docker

### Intermediate → Advanced (2-4 weeks)
1. Add authentication & RBAC to servers
2. Build complex multi-server orchestrations
3. Implement caching and performance optimization
4. Deploy to cloud (Azure Container Instances)

### Advanced (4+ weeks)
1. Async/streaming MCP patterns
2. Large-scale MCP infrastructure
3. Custom MCP SDKs in other languages
4. Contribute to MCP ecosystem

## Resources

- **Official MCP Docs**: https://modelcontextprotocol.io/
- **GitHub Repositories**: Search "mcp-server-" for implementations
- **Community**: Discord, Reddit /r/OpenAI, HackerNews

## Key Takeaways

1. **MCP = Universal API Standard for AI** - Same as how HTTP unified web
2. **Agents > Models** - MCP enables agents to take real actions
3. **Security First** - Always implement proper auth, RBAC, logging
4. **Start Simple** - 1 tool, 1 resource, then grow
5. **Leverage Pre-built** - Don't reinvent, use existing servers

---

## 🎯 Practical Challenge

**Try This**:
1. Create your own MCP server for a CSV or JSON file
2. Expose "search" and "aggregate" tools
3. Connect it to an agent
4. Run a workflow that queries your data

**Expected Time**: 1-2 hours

---

**Congratulations! You now understand MCP servers and how they power agentic AI! 🚀**
